# 分步

In [1]:
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

True

## Load

In [2]:
from langchain_community.document_loaders import TextLoader

file_path = "./data/translated_article.txt"
loader = TextLoader(file_path, encoding='utf-8')
docs = loader.load()

In [3]:
len(docs)

1

In [4]:
docs

[Document(page_content='# 提示工程\n\n日期：2023年3月15日 | 预计阅读时间：21分钟 | 作者：Lilian Weng\n\n**提示工程**（Prompt Engineering），亦称**上下文提示**（In-Context Prompting），涉及如何与大语言模型（LLM）进行交流，通过不更新模型权重的方式引导其行为，以获得预期的结果。这是一门基于实验的科学，不同模型之间提示工程的效果可能大相径庭，因此需要大量的试验和启发式方法。\n\n本文专注于自回归语言模型的提示工程，不包括完形填空测试、图像生成或多模态模型。提示工程的核心目标是实现模型的对齐和可控制性。详情可参阅我之前关于可控文本生成的[文章](https://lilianweng.github.io/posts/2021-01-02-controllable-text-generation/)。\n\n[我个人的辛辣观点] 在我看来，一些关于提示工程的论文没必要长达8页，因为这些技巧可以简单地在一句话或几句话内阐明，剩下的大都是基准测试。一个易于使用且共享的基准测试设施对社区更有益。而设置迭代提示或使用外部工具并非易事，让整个研究界接受这一做法也同样困难。\n\n# 基础提示\n\n零样本（Zero-shot）和少样本（Few-shot）学习是两种基本的模型提示方法，这两种方法最早由多篇大语言模型论文提出，并广泛用于评估大语言模型的性能。\n\n## 零样本\n\n**零样本学习**指的是直接将任务文本输入模型并询问结果。\n\n（所有情感分析示例均来源于SST-2）\n\n```\n文本: 我敢打赌视频游戏比电影有趣多了。\n情感:\n```\n\n## 少样本\n\n**少样本学习**通过提供一系列高质量的示例，每个示例都包含目标任务的输入和期望输出，帮助模型更好地理解人类的意图和评价标准。因此，少样本学习通常比零样本学习有更好的表现。但这种方法会消耗更多的Token，并且当输入和输出文本较长时可能触及上下文长度的限制。\n\n```\n文本: （劳伦斯在舞台上）到处跳跃，跳舞，跑步，出汗，擦脸，充分展现了他初次成名时的惊人才华。\n情感: 正面\n\n文本: 尽管所有迹象都表明相反，这部糟糕的电影还是设法冒充了一部正式的商业影片，收取全价门票，在电视上大肆

## Split

参考：[MarkdownHeaderTextSplitter](https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/markdown_header_metadata/)

In [5]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False
)
markdown_text = "\n\n".join([doc.page_content for doc in docs])

md_header_splits = markdown_splitter.split_text(markdown_text)

In [6]:
len(md_header_splits)

4

In [7]:
md_header_splits

[Document(page_content='# 提示工程  \n日期：2023年3月15日 | 预计阅读时间：21分钟 | 作者：Lilian Weng  \n**提示工程**（Prompt Engineering），亦称**上下文提示**（In-Context Prompting），涉及如何与大语言模型（LLM）进行交流，通过不更新模型权重的方式引导其行为，以获得预期的结果。这是一门基于实验的科学，不同模型之间提示工程的效果可能大相径庭，因此需要大量的试验和启发式方法。  \n本文专注于自回归语言模型的提示工程，不包括完形填空测试、图像生成或多模态模型。提示工程的核心目标是实现模型的对齐和可控制性。详情可参阅我之前关于可控文本生成的[文章](https://lilianweng.github.io/posts/2021-01-02-controllable-text-generation/)。  \n[我个人的辛辣观点] 在我看来，一些关于提示工程的论文没必要长达8页，因为这些技巧可以简单地在一句话或几句话内阐明，剩下的大都是基准测试。一个易于使用且共享的基准测试设施对社区更有益。而设置迭代提示或使用外部工具并非易事，让整个研究界接受这一做法也同样困难。', metadata={'Header 1': '提示工程'}),
 Document(page_content='# 基础提示  \n零样本（Zero-shot）和少样本（Few-shot）学习是两种基本的模型提示方法，这两种方法最早由多篇大语言模型论文提出，并广泛用于评估大语言模型的性能。', metadata={'Header 1': '基础提示'}),
 Document(page_content='## 零样本  \n**零样本学习**指的是直接将任务文本输入模型并询问结果。  \n（所有情感分析示例均来源于SST-2）  \n```\n文本: 我敢打赌视频游戏比电影有趣多了。\n情感:\n```', metadata={'Header 1': '基础提示', 'Header 2': '零样本'}),
 Document(page_content='## 少样本  \n**少样本学习**通过提供一系列高质量的示例，每个示例都包含目标任务的输入和期望输出，帮助模型更好地理解人类的意图和评价标准。因此，

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 500
chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

splits = text_splitter.split_documents(md_header_splits)

In [9]:
len(splits)

6

In [10]:
splits

[Document(page_content='# 提示工程  \n日期：2023年3月15日 | 预计阅读时间：21分钟 | 作者：Lilian Weng  \n**提示工程**（Prompt Engineering），亦称**上下文提示**（In-Context Prompting），涉及如何与大语言模型（LLM）进行交流，通过不更新模型权重的方式引导其行为，以获得预期的结果。这是一门基于实验的科学，不同模型之间提示工程的效果可能大相径庭，因此需要大量的试验和启发式方法。  \n本文专注于自回归语言模型的提示工程，不包括完形填空测试、图像生成或多模态模型。提示工程的核心目标是实现模型的对齐和可控制性。详情可参阅我之前关于可控文本生成的[文章](https://lilianweng.github.io/posts/2021-01-02-controllable-text-generation/)。', metadata={'Header 1': '提示工程'}),
 Document(page_content='[我个人的辛辣观点] 在我看来，一些关于提示工程的论文没必要长达8页，因为这些技巧可以简单地在一句话或几句话内阐明，剩下的大都是基准测试。一个易于使用且共享的基准测试设施对社区更有益。而设置迭代提示或使用外部工具并非易事，让整个研究界接受这一做法也同样困难。', metadata={'Header 1': '提示工程'}),
 Document(page_content='# 基础提示  \n零样本（Zero-shot）和少样本（Few-shot）学习是两种基本的模型提示方法，这两种方法最早由多篇大语言模型论文提出，并广泛用于评估大语言模型的性能。', metadata={'Header 1': '基础提示'}),
 Document(page_content='## 零样本  \n**零样本学习**指的是直接将任务文本输入模型并询问结果。  \n（所有情感分析示例均来源于SST-2）  \n```\n文本: 我敢打赌视频游戏比电影有趣多了。\n情感:\n```', metadata={'Header 1': '基础提示', 'Header 2': '零样本'}),
 Document(page_content='## 少样本  \n**少样本学习**通过

## Embed

In [11]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings()

## Store

In [12]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)

## Retrieve

In [13]:
retriever = vectorstore.as_retriever()

## Generate

In [15]:
from langchain import hub
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# prompt = hub.pull("rlm/rag-prompt")
system_prompt = (
    "你是一个用于问答任务的助手。"
    "使用以下检索到的上下文片段来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "答案最多使用三句话，并保持简洁。"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)

llm = ChatOpenAI()
output_parser = StrOutputParser()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

In [16]:
question = "什么是few shot？"

In [17]:
result = rag_chain.invoke(question)
print(result)

少样本学习（Few-shot learning）是一种模型提示方法，通过提供少量高质量示例来帮助模型更好地理解任务，通常比零样本学习表现更好，但会消耗更多Token并可能受到上下文长度限制。


# 整体

In [18]:
from dotenv import load_dotenv
from langchain import hub
from langchain_community.document_loaders import TextLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 加载环境变量
load_dotenv()

# Load
file_path = "./data/translated_article.txt"
loader = TextLoader(file_path, encoding='utf-8')
docs = loader.load()

# Split
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on, strip_headers=False
)
markdown_text = "\n\n".join([doc.page_content for doc in docs])

md_header_splits = markdown_splitter.split_text(markdown_text)

chunk_size = 500
chunk_overlap = 50
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size, chunk_overlap=chunk_overlap
)

splits = text_splitter.split_documents(md_header_splits)

# Embed
embedding = OpenAIEmbeddings()

# Store
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding)

# Retrieve
retriever = vectorstore.as_retriever()

# Generate

# Prompt
# prompt = hub.pull("rlm/rag-prompt")
system_prompt = (
    "你是一个用于问答任务的助手。"
    "使用以下检索到的上下文片段来回答问题。"
    "如果你不知道答案，就说你不知道。"
    "答案最多使用三句话，并保持简洁。"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{question}"),
    ]
)

# LLM
llm = ChatOpenAI()

# Output Parser
output_parser = StrOutputParser()

# Chain
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

# Run
question = "什么是few shot？"
print(f"Q: {question}\nA: ")

result = rag_chain.invoke(question)
print(result)

Q: 什么是few shot？
A: 
Few-shot学习通过提供少量示例来帮助模型更好地理解人类意图和评价标准，通常比零样本学习表现更好。然而，这种方法可能需要更多的Token，并且在处理长文本时可能受到上下文长度限制。
